# Preprocesamiento dentro del entrenamiento

Imputación, escalado, poda y reducción tienen que aprenderse con la parte de entrenamiento de cada partición, incluidas las interiores.

## Inicialización

Usamos la misma identidad de datos, código y protocolo en todos los cuadernos.

In [1]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
while not (ROOT/'scripts/mlcore').is_dir():
    if ROOT == ROOT.parent: raise RuntimeError('No se encontro la raiz del proyecto')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'scripts'))
from mlcore.experiment import initialize, aligned
from mlcore.data import read_data, FOLDS, TEST
OUT = initialize()
print('Ejecucion verificada:', OUT.name)


Ejecucion verificada: 01afe756130bafc6


**Qué encontramos y cómo se usa.** La identidad evita mezclar artefactos de protocolos distintos. Los cálculos científicos viven en `scripts/mlcore`; estos cuadernos explican, ejecutan y muestran cada etapa.

## Medianas y esquema de columnas

Mostramos un ajuste de demostración limitado a las primeras dos temporadas. No es un ajuste global reutilizado en validación.

In [2]:
from mlcore.models import preprocessing
df, sets = read_data(OUT)
tr = df[df.Season.lt(1617)]
for branch in ['con_cuotas','sin_cuotas']:
    pre = preprocessing(); X = pre.fit_transform(tr[sets[branch]].to_numpy(float), tr.y.to_numpy())
    print(branch, '| entrada:', len(sets[branch]), '| salida tras poda:', X.shape[1], '| filas:', len(tr))
    assert np.isfinite(X).all()

con_cuotas | entrada: 84 | salida tras poda: 79 | filas: 5968
sin_cuotas | entrada: 77 | salida tras poda: 74 | filas: 5968


/home/evssz/UTEC/6to/machine/ML_Project/scripts/mlcore/data.py:290: DtypeWarning: Columns (0: Time, 1: B365CH) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(out/'dataset.csv', parse_dates=['Date', 'H_snapshot_at', 'A_snapshot_at'])


**Qué encontramos / decisión.** Los faltantes se resuelven dentro de cada entrenamiento, conservando columnas completamente vacías e indicadores. No se consulta el test para decidir F1/F2.

## MI frente a PCA

Se registran las alternativas antes de evaluar: MI-5, MI-10, PCA-5 y PCA-95%, con la misma familia LR y búsqueda de C en ambas ramas.

In [3]:
from mlcore.models import specifications, GRIDS
display(pd.DataFrame([{'enfoque':s.branch,'modelo':s.name,'familia':s.family,'reduccion':s.reduction,'n_variables':len(s.columns),'configuraciones':len(GRIDS[s.family])} for s in specifications(sets)]))

,enfoque,modelo,familia,reduccion,n_variables,configuraciones
0,con,LR,LR,all,84,3
1,con,HGB,HGB,all,84,3
2,con,XGB,XGB,all,84,3
3,con,RF,RF,all,84,3
4,con,MLP,MLP,all,84,3
5,con,SVM,SVM,all,84,3
6,con,LR_MI5,LR,MI5,84,3
7,con,LR_MI10,LR,MI10,84,3
8,con,LR_PCA5,LR,PCA5,84,3
9,con,LR_PCA95,LR,PCA95,84,3


**Qué encontramos / decisión.** Los resultados se generarán en 04 y se interpretarán en 05. No se elige la mejor representación usando la temporada que se predice.

## Pruebas contra fuga

Las pruebas alteran resultados y ratings futuros y exigen que las variables del pasado no cambien.

In [4]:
import subprocess
r = subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=ROOT,capture_output=True,text=True)
print(r.stdout+r.stderr)
r.check_returncode()

test_bootstrap_week_includes_iso_year (test_temporal_integrity.TemporalTests.test_bootstrap_week_includes_iso_year) ... ok
test_equal_mass_ece_handles_constant_probabilities (test_temporal_integrity.TemporalTests.test_equal_mass_ece_handles_constant_probabilities) ... ok
test_future_roster_cannot_change_past_features (test_temporal_integrity.TemporalTests.test_future_roster_cannot_change_past_features) ... ok
test_future_rows_do_not_change_normalization (test_temporal_integrity.TemporalTests.test_future_rows_do_not_change_normalization) ... ok
test_future_scores_cannot_change_past_features (test_temporal_integrity.TemporalTests.test_future_scores_cannot_change_past_features) ... ok
test_imputer_only_learns_training_rows (test_temporal_integrity.TemporalTests.test_imputer_only_learns_training_rows) ... ok
test_invalid_odds_fail_clearly (test_temporal_integrity.TemporalTests.test_invalid_odds_fail_clearly) ... ok
test_market_quality_cutoff_and_partial_books (test_temporal_integrity.Tempo

**Qué encontramos / decisión.** Se comprueban también ECE constante, cuotas inválidas, separación del SVM y bloques ISO. La unión cartesiana del calentamiento desaparece del nuevo cálculo.